# ASR & STT Fundamentals
## From Sound Waves to Digital Text

**Course**: Programming for Linguistics (PfL 2026)
**Instructor**: Luca Ducceschi

### Overview:
1. **Digital Audio Signal Processing**: Analog-to-Digital Conversion (ADC).
2. **Feature Extraction**: Turning audio into "images" (Mel Spectrograms).
3. **Architectures**: Encoder-Decoder Transformers (Whisper) & Self-Supervision (wav2vec).
4. **Decoding**: Attention mechanisms and sequence generation.
5. **Advanced**: Fine-tuning Whisper with PyTorch Lightning.

## 1. Digital Audio Signal Processing

### Analog vs. Digital
- **Analog**: Continuous sound waves (changes in air pressure).
- **Digital**: Discrete representation of the sound wave.

### The ADC Process
1. **Sampling**: Measuring the amplitude of the signal at regular intervals (Sample Rate, e.g., 16kHz).
2. **Quantization**: Mapping the continuous amplitude values to a finite set of digital values (Bit Depth, e.g., 16-bit).

### Nyquist-Shannon Sampling Theorem
To accurately reconstruct a signal, the sampling rate must be at least **twice** the highest frequency component (Nyquist frequency).
- Human hearing: ~20Hz to 20kHz.
- CD Quality: 44.1kHz.
- Standard Speech AI: **16kHz** (captures up to 8kHz, enough for most linguistic info).

In [1]:
# 1. Fully remove any cached or broken builds
# ! pip uninstall torch torchaudio -y

# 2. Reinstall with matching versions
# !pip install torch==2.6.0 torchaudio==2.6.0 --no-cache-dir


[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# from torch import (
#     device as _device,

# )
import torch
import torchaudio
import matplotlib.pyplot as plt
import os
import IPython
from torchaudio.utils import _download_asset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
SPEECH_FILE = _download_asset("tutorial-assets/Lab41-SRI-VOiCES-src-sp0307-ch127535-sg0042.wav")

waveform, sr = torchaudio.load(SPEECH_FILE)
print(f"Original Sample Rate: {sr} Hz")

resampler = torchaudio.transforms.Resample(sr, 16000)
waveform_16k = resampler(waveform)
print(f"New Sample Rate: 16000 Hz")

ImportError: cannot import name '_download_asset' from 'torchaudio.utils' (/Users/luca/Envs/hfwhsp/lib/python3.13/site-packages/torchaudio/utils/__init__.py)

In [2]:
# pip install torch==2.11.0 torchaudio==2.11.0

  Using cached torch-2.11.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (29 kB)
  Using cached torchaudio-2.11.0-cp313-cp313-macosx_12_0_arm64.whl.metadata (6.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 MB 2.8 MB/s eta 0:00:0000:0100:01
Using cached torchaudio-2.11.0-cp313-cp313-macosx_12_0_arm64.whl (679 kB)

[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
IPython.display.Audio(SPEECH_FILE)

NameError: name 'IPython' is not defined

In [ ]:
import matplotlib.pyplot as plt
import librosa.display
array, sampling_rate = librosa.load(SPEECH_FILE)
plt.figure().set_figwidth(12)
librosa.display.waveshow(array, sr=sampling_rate)

## 2. Feature Extraction: Audio as Images

Raw audio is a 1D sequence of amplitudes. Transformers prefer 2D "image-like" representations.

### The Path to the Mel Spectrogram
1. **STFT (Short-Time Fourier Transform)**: We take small windows (frames) of audio and calculate the frequency content (FFT).
2. **Magnitude Spectrogram**: Squaring the FFT results.
3. **Mel Scaling**: Humans perceive pitch logarithmically. The Mel Scale compresses higher frequencies where we are less sensitive.
4. **Log Compression**: Humans perceive loudness logarithmically (dB).

In [ ]:
import whisper

# Whisper's built-in processing
audio = whisper.pad_or_trim(waveform_16k.flatten())
mel = whisper.log_mel_spectrogram(audio)

plt.figure(figsize=(10, 4))
plt.imshow(mel.numpy(), aspect='auto', origin='lower')
plt.title("Log-Mel Spectrogram (Whisper Input)")
plt.xlabel("Frames")
plt.ylabel("Mel Filter")
plt.show()

## 3. The Architecture: Whisper

Whisper is an **Encoder-Decoder Transformer** trained on 680,000 hours of labeled data.

### The Encoder
- **CNN Layers**: Two convolutional layers for downsampling (reducing the time dimension).
- **Transformer Blocks**: Multi-head self-attention layers that learn global context from the spectrogram.

### The Decoder
- **Autoregressive**: It predicts text one token at a time.
- **Cross-Attention**: The bridge between audio features and text generation. The decoder "attends" to the relevant parts of the encoder's output.

![Whisper Architecture](file:///Users/luca/.gemini/antigravity/brain/d080b50e-76b2-4b34-8dc4-2a8f908418a7/whisper_architecture_diagram_1777533865091.png)

## 4. wav2vec 2.0: Self-Supervised Learning

Unlike Whisper, wav2vec 2.0 learns from **unlabeled** audio first.

1. **Latent Representation**: Encodes raw audio into a sequence of vectors.
2. **Quantization**: Converts these vectors into discrete "codebook" entries.
3. **Contrastive Task**: Predicts masked speech units. This forces the model to understand the structure of language without knowing the words.

**Fine-tuning**: We then add a simple linear layer on top to map these units to characters/words (CTC Loss).

## 5. Advanced: Fine-tuning Whisper

In your `augusta_code` folder, you have `wishtune.py`. This script implements an efficient fine-tuning pipeline.

### Key Optimization Strategy
**Freezing the Encoder**: In `wishtune.py`, we only train the decoder. 
```python
for p in self.model.encoder.parameters():
    p.requires_grad = False
```
Why? Because the encoder already knows how to "hear" speech perfectly. We only want to teach the decoder a specific dialect or technical vocabulary.

### Training Loop with PyTorch Lightning

The training step in `wishtune.py` looks like this:

```python
def training_step(self, batch, batch_id):
    input_ids = batch["input_ids"] # Spectrogram
    labels = batch["labels"].long() # Target text
    dec_input_ids = batch["dec_input_ids"].long() # Current context

    # 1. Get audio features (No gradients here!)
    with torch.no_grad():
        audio_features = self.model.encoder(input_ids)
    
    # 2. Pass to decoder
    out = self.model.decoder(dec_input_ids, audio_features)
    
    # 3. Calculate Loss (Cross Entropy)
    loss = self.loss_fn(out.view(-1, out.size(-1)), labels.view(-1))
    return loss
```

## 6. Performance Metrics

How do we know if our model is good?

1. **WER (Word Error Rate)**: Substitutions + Deletions + Insertions / Total Words.
2. **CER (Character Error Rate)**: Same as WER, but at the character level.
3. **BLEU**: Measures n-gram overlap.

### Practical Exercise
Go to the `augusta_code` folder and check `train_augusta_optuna.py`. It uses **Optuna** to automatically find the best hyperparameters (learning rate, warmup steps) for your specific dataset!